# Hybrid retrieval: global + same-location

Финальная конфигурация читается из `hybrid_config.json`. Ноутбук воспроизводит метрики на готовых кешах, не выполняет grid search и не записывает ответ.

## 1. Setup

In [2]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.validation import load_validation
from src.pipeline import retrieve, retrieve_sources, item_locations, fuse
from src.metrics import evaluate
from src.fusion import reciprocal_rank_fusion

CORPUS, CACHE = ROOT / 'data/train.parquet', ROOT / '.cache'
queries, truth = load_validation(ROOT, 'quick')
print(f'Quick validation: {len(queries)} contexts')

Quick validation: 2452 contexts


In [3]:
config = json.loads((ROOT / 'hybrid_config.json').read_text())
confirm_queries, confirm_truth = load_validation(ROOT, 'confirmation')
assert not set(queries.index) & set(confirm_queries.index)
print(f'Confirmation: {len(confirm_queries)} contexts; overlap with quick: 0')

Confirmation: 2452 contexts; overlap with quick: 0


## 2. Sparse sources

Глобальные word и char TF-IDF возвращают top-500. Word сопоставляет запрос с заголовком и параметрами, char - с заголовком.

In [5]:
sparse_config = {**config, 'sources': config['sources'][:2]}
results = retrieve_sources(CORPUS, queries, sparse_config, CACHE, cache_only=True)
item_ids = results['word'].item_ids
locations = item_locations(CORPUS, item_ids)

## 3. Dense E5 retrieval

`intfloat/multilingual-e5-small` ищет по заголовкам. Используются готовые нормированные embeddings и top-500 выдача.

In [7]:
results['dense'] = retrieve(
    CORPUS, queries, config['sources'][2], CACHE,
    model_info=config['e5_model'], top_k=config['top_k_per_source'], cache_only=True)
np.testing.assert_array_equal(item_ids, results['dense'].item_ids)
assert results['dense'].positions.shape == (len(queries), 500)
assert np.isfinite(results['dense'].scores).all()
print('Dense top-500 loaded; item-ID alignment checked')

Dense top-500 loaded; item-ID alignment checked


In [8]:
from src.dense_retrieval import check_cached_embeddings
from src.text import query_texts
display(check_cached_embeddings(
    CORPUS, query_texts(queries, 'query_only'), CACHE / 'e5', config['e5_model']))

{'sample_items': 1000,
 'sample_queries': 3,
 'dimension': 384,
 'dtype': 'float32',
 'checks': 'passed'}

## 4. Global candidates

Сравниваем глобальные источники и их union. Candidate recall - доля релевантных объявлений, попавших хотя бы в один источник до финальной сортировки.

In [10]:
rows = []
for labels in [('word',), ('char',), ('dense',), ('word', 'char'), ('word', 'char', 'dense')]:
    sources = [results[label].source for label in labels]
    positions = sources[0][0] if len(sources) == 1 else reciprocal_rank_fusion(sources)
    rows.append(dict(source='+'.join(labels), **evaluate(positions, truth, item_ids, sources)))
display(pd.DataFrame(rows).set_index('source').round(5))

,R@10,R@20,R@50,candidate_recall
source,,,,
word,0.06866,0.09838,0.16413,0.43305
char,0.07662,0.10572,0.17542,0.49431
dense,0.08177,0.11408,0.18195,0.47724
word+char,0.07209,0.10475,0.18100,0.56850
word+char+dense,0.08088,0.11582,0.19781,0.62683


## 5. Same-location retrieval

Word и char выполняют отдельный поиск среди объявлений той же локации и возвращают top-100. Это добавляет кандидатов вне глобального top-500.

In [12]:
from src.metrics import candidate_recall
for spec in config['sources'][3:]:
    name = spec['name']
    results[name] = retrieve(CORPUS, queries, spec, CACHE, model_info=config['e5_model'],
                             top_k=spec['top_k'], cache_only=True)
    np.testing.assert_array_equal(item_ids, results[name].item_ids)
global_sources = [results[name].source for name in ['word', 'char', 'dense']]
all_sources = [results[spec.get('name', spec['kind'])].source for spec in config['sources']]
display(pd.DataFrame([
    {'sources': 'global', 'candidate_recall': candidate_recall(global_sources, truth, item_ids)},
    {'sources': 'global + same-location', 'candidate_recall': candidate_recall(all_sources, truth, item_ids)}
]).round(5))

,sources,candidate_recall
0,global,0.62683
1,global + same-location,0.83255


## 6. Weighted fusion and location

Пять источников объединяются по RRF: `weight / (60 + rank)`. Мягкий бонус локации переставляет найденных кандидатов; локальные источники дополнительно расширяют сам union.

In [14]:
display(pd.DataFrame([
    {'source': spec.get('name', spec['kind']), 'top_k': spec.get('top_k', config['top_k_per_source']),
     'weight': weight}
    for spec, weight in zip(config['sources'], config['weights'])
]))
print('RRF C:', config['rrf_constant'], 'location boost:', config['location_weight'])

RRF C: 60 location boost: 0.05


,source,top_k,weight
0,word,500,1.00
1,char,500,0.75
2,dense,500,1.75
3,local_char,100,0.25
4,local_word,100,0.25


In [15]:
ablation = []
for boost in [0., config['location_weight']]:
    positions = fuse(results, queries, {**config, 'location_weight': boost}, locations)
    ablation.append(dict(location=boost,
                        **evaluate(positions, truth, item_ids, [r.source for r in results.values()])))
display(pd.DataFrame(ablation).set_index('location').round(5))
for metric, expected in config['quick_metrics'].items():
    assert abs(ablation[-1][metric] - expected) < 1e-5, (metric, ablation[-1][metric])

,R@10,R@20,R@50,candidate_recall
location,,,,
0.00,0.11788,0.16231,0.25652,0.83255
0.05,0.54631,0.61088,0.69080,0.83255


## 7. Confirmation

Confirmation содержит 2452 других контекста. На ней проверялся короткий список настроек; это не нетронутый test.

In [17]:
confirm_results = retrieve_sources(
    CORPUS, confirm_queries, config, CACHE,
    search_dir=CACHE / 'hybrid/confirm', cache_only=True)
np.testing.assert_array_equal(item_ids, confirm_results['word'].item_ids)
old_sparse = {**config, 'sources': config['sources'][:2],
              'weights': [1., 1.5], 'location_weight': .01}
confirmation = []
for name, settings in [('old sparse', old_sparse), ('final hybrid', config)]:
    positions = fuse(confirm_results, confirm_queries, settings, locations)
    sources = [confirm_results[s.get('name', s['kind'])].source for s in settings['sources']]
    confirmation.append(dict(pipeline=name, **evaluate(positions, confirm_truth, item_ids, sources)))
display(pd.DataFrame(confirmation).set_index('pipeline').round(5))
for metric, expected in config['confirmation_metrics'].items():
    assert abs(confirmation[-1][metric] - expected) < 1e-5

,R@10,R@20,R@50,candidate_recall
pipeline,,,,
old sparse,0.14315,0.19594,0.31114,0.56531
final hybrid,0.54012,0.60263,0.68559,0.84021


## 8. Final configuration

Лучший отправленный leaderboard Recall@50 - **0.678731**. У финального конфига quick Recall@50 = **0.690799**, confirmation Recall@50 = **0.685595**, confirmation candidate recall = **0.840209**.

In [19]:
display(pd.DataFrame([
    {**spec, 'weight': weight} for spec, weight in zip(config['sources'], config['weights'])
]))
print({key: config[key] for key in ['rrf_constant', 'top_k_per_source', 'location_weight']})
display(pd.DataFrame([
    dict(source=name, index_MiB=r.index_mib, load_index_s=r.build_or_load_s,
         load_candidates_s=r.search_or_load_s, cache_hit=r.cache_hit)
    for name, r in results.items()
]).set_index('source').round(3))

{'rrf_constant': 60, 'top_k_per_source': 500, 'location_weight': 0.05}


,kind,document,query,weight,scope,name,top_k
0,word,title_params,query_only,1.00,NaN,NaN,NaN
1,char,title,query_only,0.75,NaN,NaN,NaN
2,dense,title,query_only,1.75,NaN,NaN,NaN
3,char,title,query_only,0.25,same_location,local_char,100.0
4,word,title_params,query_only,0.25,same_location,local_word,100.0


,index_MiB,load_index_s,load_candidates_s,cache_hit
source,,,,
word,412.862,4.778,0.100,True
char,185.269,3.168,0.102,True
dense,505.115,1.833,0.084,True
local_char,185.269,2.898,0.735,None
local_word,412.862,5.749,1.294,None


## 9. Воспроизведение ответа

Из корня проекта: `python run.py --config hybrid_config.json --output answer.csv`. Контрольная SHA-256 приведена в README. Ячейка проверяет CSV, если он уже создан.

In [21]:
from src.submission import check_submission
answer_path = ROOT / "answer.csv"
if answer_path.exists():
    display(check_submission(answer_path, ROOT / "data/benchmark_queries.parquet",
                             ROOT / "data/benchmark_items.parquet"))
else:
    print("Сначала выполните python run.py --config hybrid_config.json --output answer.csv")

{'queries': 2452, 'candidates_per_query': [50]}